## IMPORTS

In [9]:
# Auto reload modules
%load_ext autoreload
%autoreload all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import numpy as np
import pandas as pd
import os
from tqdm.notebook import tqdm

from weac.components import Segment, ScenarioConfig, WeakLayer, CriteriaConfig, WEAK_LAYER
from weac.analysis import CriteriaEvaluator
from weac.utils.snowpilot_parser import convert_to_mm, convert_to_deg

from layerwise.analysis.profile_utils import load_snowpilot_parsers, eval_avalanche_pit

## SETTINGS

In [11]:
## Settings
run_weac = True
dev = True
discard_pits_with_wl_above = 50 # mm

## Setup standard values

raw_data_dir = "../data/raw/slf-ect"
csv_file = "../data/misc/weac_over_slf_ect.csv"


## PARSE SNOWPYLOT

In [12]:
## Parse all snowpilot files near an avalanche and a layer of concern

paths, parsers = load_snowpilot_parsers(raw_data_dir)

print(f"Overall number of files: {len(paths)}")

paths_and_parsers = [
    (fp, pars)
    for fp, pars in zip(paths, parsers)
]

Overall number of files: 6533


In [13]:
different_ect_scores = set(
    ect.test_score
    for fp, pars in paths_and_parsers
    for ect in pars.snowpit.stability_tests.ECT
)

print(len(paths_and_parsers))
print(different_ect_scores)
print(len(different_ect_scores))

6533
{'ECTP26', 'ECTP17', 'ECTP27', 'ECTN29', 'ECTP21', 'ECTP8', 'ECTP9', 'ECTP18', 'ECTP11', 'ECTP29', 'ECTN3', 'ECTN8', 'ECTN30', 'ECTP28', None, 'ECTN4', 'ECTP19', 'ECTN2', 'ECTP6', 'ECTP22', 'ECTN20', 'ECTN19', 'ECTN21', 'ECTN26', 'ECTP20', 'ECTN16', 'ECTN15', 'ECTP1', 'ECTN9', 'ECTN28', 'ECTP7', 'ECTP13', 'ECTP10', 'ECTN13', 'ECTN12', 'ECTN14', 'ECTN22', 'ECTP5', 'ECTN7', 'ECTP3', 'ECTN6', 'ECTP15', 'ECTN11', 'ECTP2', 'ECTN23', 'ECTN25', 'ECTN1', 'ECTP25', 'ECTN27', 'ECTP14', 'ECTN5', 'ECTP12', 'ECTP23', 'ECTP24', 'ECTP30', 'ECTPV', 'ECTP16', 'ECTN10', 'ECTN24', 'ECTN18', 'ECTN17', 'ECTP4'}
62


## EXTRACT INFO

Extract Depth at which the weak layer should be inserted.

In this case: Depth of ECT Result

In [ ]:
# Extract additional pit information

pit_info_list = []
unique_ect_entries = []
for i,(fp, pit) in enumerate(paths_and_parsers):
    if i % 100 == 0:
        print(f"Processing pit {i} of {len(paths_and_parsers)}")
    for ect in pit.snowpit.stability_tests.ECT:
        try:
            hs = pit.snowpit.snow_profile.hs
            if hs:
                hs_mm = hs[0] * convert_to_mm[hs[1]]
            else:
                hs_mm = None
            profile_depth = pit.snowpit.snow_profile.profile_depth
            if profile_depth:
                profile_depth_mm = profile_depth[0] * convert_to_mm[profile_depth[1]]
            else:
                profile_depth_mm = None
            depth_top_mm = ect.depth_top[0] * convert_to_mm[ect.depth_top[1]]
            test_score = ect.test_score
            slope_angle = pit.snowpit.core_info.location.slope_angle
            if slope_angle:
                slope_angle_deg = slope_angle[0] * convert_to_deg[slope_angle[1]]
            else:
                slope_angle_deg = 0.0
            pit_info_dict = {
                "Slope Angle": slope_angle_deg,
                "HS": hs_mm,
                "Profile Depth": profile_depth_mm,
                "WL_Depth": depth_top_mm,
                "ECT_Score": test_score,
                "ECT_Propagation": ect.propagation,
                "ECT_NumTaps": ect.num_taps,
            }
            unique_ect_entries.append(pit)
            pit_info_list.append(pit_info_dict)
        except Exception as e:
            print(f"Skipping pit {fp}")
print(len(unique_ect_entries))

Processing pit 0 of 6533

	 depth_top: [10.0, 'cm']
	 test_score: ECTN11
	 comment: None
	 propagation: False
	 num_taps: 11

	 depth_top: [10.0, 'cm']
	 test_score: ECTN1
	 comment: None
	 propagation: False
	 num_taps: 1

	 depth_top: [27.0, 'cm']
	 test_score: ECTP6
	 comment: None
	 propagation: True
	 num_taps: 6

	 depth_top: [27.0, 'cm']
	 test_score: ECTP5
	 comment: None
	 propagation: True
	 num_taps: 5

	 depth_top: None
	 test_score: None
	 comment: None
	 propagation: None
	 num_taps: None
Skipping pit ../data/raw/slf-ect/10f3ada0-407a-439b-b67f-3bc990f011ca.caaml.xml

	 depth_top: [100.0, 'cm']
	 test_score: ECTP30
	 comment: None
	 propagation: True
	 num_taps: 30

	 depth_top: [78.0, 'cm']
	 test_score: ECTP2
	 comment: None
	 propagation: True
	 num_taps: 2

	 depth_top: [78.0, 'cm']
	 test_score: ECTP21
	 comment: None
	 propagation: True
	 num_taps: 21

	 depth_top: [78.0, 'cm']
	 test_score: ECTP15
	 comment: None
	 propagation: True
	 num_taps: 15

	 depth_top: [70

## RUN WEAC

In [15]:


phi = 35.0
scenario_config = ScenarioConfig(system_type="skier", phi=phi)
weak_layer = WEAK_LAYER
segments = [
    Segment(length=10000, has_foundation=True, m=0.0),
    Segment(
        length=10000,
        has_foundation=True,
        m=0.0,
    ),
]
criteria_config = CriteriaConfig()
criteria_evaluator = CriteriaEvaluator(criteria_config)

In [17]:
if run_weac:
    data_rows = []
    error_pits = []
    entries = list(zip(unique_ect_entries, pit_info_list))
    if dev:
        entries = entries[:5]
    for pit, pit_info_dict in tqdm(
        entries,
        total=len(entries),
        desc="Processing pits",
    ):
        scenario_config = ScenarioConfig(
            phi=pit_info_dict["Slope Angle"],
            system_type="skier",
        )
        try:
            pit_info_dict, layers, weaklayer = eval_avalanche_pit(
                pit, 
                pit_info_dict, 
                scenario_config, 
                segments, 
                weak_layer,
                criteria_evaluator)
            data_rows.append(pit_info_dict)
        except Exception as e:
            error_pits.append(pit)
            print(f"Error processing pit {pit.snowpit.core_info.pit_id}: {e}")
            continue

    print(f"Number of pits with errors: {len(error_pits)}")
    print(f"Number of pits processed: {len(data_rows)}")
    print(f"Number of pits total: {len(unique_ect_entries)}")

    df = pd.DataFrame(data_rows)
    print(df.head())
    df.to_csv(csv_file, index=False)
else:
    df = pd.read_csv(csv_file)

Processing pits:   0%|          | 0/5 [00:00<?, ?it/s]

Number of pits with errors: 0
Number of pits processed: 5
Number of pits total: 7391
  Slope Angle      HS  Profile Depth  WL_Depth ECT_Score  ECT_Propagation  \
0          31   410.0          410.0     100.0    ECTN11            False   
1          31   410.0          410.0     100.0     ECTN1            False   
2          30   560.0          560.0     270.0     ECTP6             True   
3          30   560.0          560.0     270.0     ECTP5             True   
4          23  1610.0         1610.0    1000.0    ECTP30             True   

  ECT_NumTaps  max_stress  impact_criterion  coupled_criterion  sserr_result  \
0          11    0.000354         13.483243          36.819231      0.443164   
1           1    0.000354         13.483243          36.819231      0.443164   
2           6    0.004892         60.313722         152.421506      1.506329   
3           5    0.004892         60.313722         152.421506      1.506329   
4          30    0.119397        129.358902         

In [18]:
sserr_median = df["sserr_result"].median()
sserr_mean = df["sserr_result"].mean()
sserr_std = df["sserr_result"].std()

print(f"SSERR Median: {sserr_median}")
print(f"SSERR Mean: {sserr_mean}")
print(f"SSERR Std: {sserr_std}")

cc_median = df["coupled_criterion"].median()
cc_mean = df["coupled_criterion"].mean()
cc_std = df["coupled_criterion"].std()

print(f"CC Median: {cc_median}")
print(f"CC Mean: {cc_mean}")
print(f"CC Std: {cc_std}")

max_stress_median = df["max_stress"].median()
max_stress_mean = df["max_stress"].mean()
max_stress_std = df["max_stress"].std()

print(f"MAX STRESS Median: {max_stress_median}")
print(f"MAX STRESS Mean: {max_stress_mean}")
print(f"MAX STRESS Std: {max_stress_std}")

ss_max_Sxx_norm_median = df["ss_max_Sxx_norm"].median()
ss_max_Sxx_norm_mean = df["ss_max_Sxx_norm"].mean()
ss_max_Sxx_norm_std = df["ss_max_Sxx_norm"].std()

print(f"SS MAX SXX NORM Median: {ss_max_Sxx_norm_median}")
print(f"SS MAX SXX NORM Mean: {ss_max_Sxx_norm_mean}")
print(f"SS MAX SXX NORM Std: {ss_max_Sxx_norm_std}")

slab_tensile_criterion_median = df["slab_tensile_criterion"].median()
slab_tensile_criterion_mean = df["slab_tensile_criterion"].mean()
slab_tensile_criterion_std = df["slab_tensile_criterion"].std()

print(f"SLAB TENSILE CRITERION Median: {slab_tensile_criterion_median}")
print(f"SLAB TENSILE CRITERION Mean: {slab_tensile_criterion_mean}")
print(f"SLAB TENSILE CRITERION Std: {slab_tensile_criterion_std}")


SSERR Median: 1.5063292246034237
SSERR Mean: 2.238156196364961
SSERR Std: 2.8746477469229332
CC Median: 152.42150606296906
CC Mean: 149.8205601834922
CC Std: 136.29481757069294
MAX STRESS Median: 0.0048919719220206544
MAX STRESS Mean: 0.02597757814567874
MAX STRESS Std: 0.05227226306899395
SS MAX SXX NORM Median: 1.9733046999056556
SS MAX SXX NORM Mean: 2.0471918366602564
SS MAX SXX NORM Std: 0.37102587518279057
SLAB TENSILE CRITERION Median: 0.2600732600732601
SLAB TENSILE CRITERION Mean: 0.26954359696154134
SLAB TENSILE CRITERION Std: 0.10676615286908926


In [19]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["WL_Depth"], y=df["impact_criterion"], mode="markers", name="Impact Criterion", marker=dict(color="red")))
fig.update_layout(xaxis_title="WL Depth (mm)", yaxis_title="Impact Criterion")
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["WL_Depth"], y=df["coupled_criterion"], mode="markers", name="Coupled Criterion", marker=dict(color="blue")))
fig.update_layout(xaxis_title="WL Depth (mm)", yaxis_title="Coupled Criterion")
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["WL_Depth"], y=df["sserr_result"], mode="markers", name="SSERR", marker=dict(color="green")))
fig.update_layout(xaxis_title="WL Depth (mm)", yaxis_title="SSERR")
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["WL_Depth"], y=df["touchdown_distance"], mode="markers", name="Touchdown Distance", marker=dict(color="yellow")))
fig.update_layout(xaxis_title="WL Depth (mm)", yaxis_title="Touchdown Distance")
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["WL_Depth"], y=df["max_stress"], mode="markers", name="Max Stress", marker=dict(color="red")))
fig.update_layout(xaxis_title="WL Depth (mm)", yaxis_title="Max Stress")
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["WL_Depth"], y=df["ss_max_Sxx_norm"], mode="markers", name="SS MAX SXX NORM", marker=dict(color="blue")))
fig.update_layout(xaxis_title="WL Depth (mm)", yaxis_title="SS MAX SXX NORM")
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["WL_Depth"], y=df["slab_tensile_criterion"], mode="markers", name="SLAB TENSILE CRITERION", marker=dict(color="green")))
fig.update_layout(xaxis_title="WL Depth (mm)", yaxis_title="SLAB TENSILE CRITERION")
fig.show()


In [ ]:
import matplotlib.pyplot as plt

# Bin wl depths according to 10 mm intervals
wl_depths = df["WL_Depth"]
max_wl_depth = max(wl_depths)
min_wl_depth = min(wl_depths)

# Create bins
bin_width = 50
bins = np.arange(min_wl_depth, max_wl_depth + bin_width, bin_width)

# Use matplotlib's histogram which handles this automatically
plt.hist(wl_depths, bins=bins, edgecolor='black', alpha=0.7)
plt.xlabel("WL Depth (mm)")
plt.ylabel("Number of Pits")
plt.title("Number of Pits in Each WL Depth Bin")
plt.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

wl_depths = df["WL_Depth"]
df = df[df["WL_Depth"] > discard_pits_with_wl_above]

### Coupled Criterion

In [ ]:
# Plot cumulative distribution of coupled criterion
cc = df["coupled_criterion"][~np.isnan(df["coupled_criterion"])]
sorted_cc = np.sort(cc)
cdf = np.arange(1, len(sorted_cc) + 1) / len(sorted_cc)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_cc, y=cdf, mode="markers", name="Coupled Criterion", marker=dict(color="red")))
fig.update_layout(xaxis_title="Weight (kg)", yaxis_title="Cumulative Distribution")
fig.show()

In [ ]:
from scipy import stats

# Fit a normal distribution to the data
params_norm = stats.norm.fit(cc)
cdf_values_norm = stats.norm.cdf(sorted_cc, *params_norm)

# Fit a log-normal distribution to the data
params_lognorm = stats.lognorm.fit(cc)
cdf_values_lognorm = stats.lognorm.cdf(sorted_cc, *params_lognorm)

# # Fit an exponential distribution to the data
# params_expon = stats.expon.fit(cc)
# cdf_values_expon = stats.expon.cdf(sorted_cc, *params_expon)

# Fit an Exponential Normal distribution to the data
params_exponnorm = stats.exponnorm.fit(cc)
cdf_values_exponnorm = stats.exponnorm.cdf(sorted_cc, *params_exponnorm)
print(params_exponnorm)


fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_cc, y=cdf_values_norm, mode="lines", name="Normal"))
fig.add_trace(go.Scatter(x=sorted_cc, y=cdf_values_lognorm, mode="lines", name="Lognormal"))
# fig.add_trace(go.Scatter(x=sorted_cc, y=cdf_values_expon, mode="lines", name="Exponential"))
fig.add_trace(go.Scatter(x=sorted_cc, y=cdf_values_exponnorm, mode="lines", name="Exponential Normal"))
fig.add_trace(go.Scatter(x=sorted_cc, y=cdf, mode="markers", name="Data"))
fig.show()


## Steady State Energy Release Rate

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Plot cumulative distribution of coupled criterion
sserr = df["sserr_result"][~np.isnan(df["sserr_result"])]
sorted_sserr = np.sort(sserr)
cdf = np.arange(1, len(sorted_sserr) + 1) / len(sorted_sserr)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_sserr, y=cdf, mode="markers", name="SSERR", marker=dict(color="red")))
fig.update_layout(xaxis_title="SSERR", yaxis_title="Cumulative Distribution")
fig.show()


In [ ]:
from scipy import stats

# # Fit a normal distribution to the data
# params_norm = stats.norm.fit(sserr)
# cdf_values_norm = stats.norm.cdf(sorted_sserr, *params_norm)

# Fit a log-normal distribution to the data
params_lognorm = stats.lognorm.fit(sserr)
cdf_values_lognorm = stats.lognorm.cdf(sorted_sserr, *params_lognorm)
print(params_lognorm)

# # Fit an exponential distribution to the data
# params_expon = stats.expon.fit(sserr)
# cdf_values_expon = stats.expon.cdf(sorted_sserr, *params_expon)

# Fit an Exponential Normal distribution to the data
params_exponnorm = stats.exponnorm.fit(sserr)
cdf_values_exponnorm = stats.exponnorm.cdf(sorted_sserr, *params_exponnorm)


fig = go.Figure()
# fig.add_trace(go.Scatter(x=sorted_sserr, y=cdf_values_norm, mode="lines", name="Normal"))
fig.add_trace(go.Scatter(x=sorted_sserr, y=cdf_values_lognorm, mode="lines", name="Lognormal"))
# fig.add_trace(go.Scatter(x=sorted_sserr, y=cdf_values_expon, mode="lines", name="Exponential"))
fig.add_trace(go.Scatter(x=sorted_sserr, y=cdf_values_exponnorm, mode="lines", name="Exponential Normal"))
fig.add_trace(go.Scatter(x=sorted_sserr, y=cdf, mode="markers", name="Data"))
fig.show()


## Max Stress

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Plot cumulative distribution of coupled criterion
max_stress = df["max_stress"][~np.isnan(df["max_stress"])]
sorted_max_stress = np.sort(max_stress)
cdf = np.arange(1, len(sorted_max_stress) + 1) / len(sorted_max_stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_max_stress, y=cdf, mode="markers", name="max_stress", marker=dict(color="red")))
fig.update_layout(xaxis_title="max_stress", yaxis_title="Cumulative Distribution")
fig.show()


In [ ]:
from scipy import stats

# # Fit a normal distribution to the data
params_norm = stats.norm.fit(max_stress)
cdf_values_norm = stats.norm.cdf(sorted_max_stress, *params_norm)

# Fit a log-normal distribution to the data
params_lognorm = stats.lognorm.fit(max_stress)
cdf_values_lognorm = stats.lognorm.cdf(sorted_max_stress, *params_lognorm)
print(params_lognorm)

# # Fit an exponential distribution to the data
params_expon = stats.expon.fit(max_stress)
cdf_values_expon = stats.expon.cdf(sorted_max_stress, *params_expon)

# Fit an Exponential Normal distribution to the data
params_exponnorm = stats.exponnorm.fit(max_stress)
cdf_values_exponnorm = stats.exponnorm.cdf(sorted_max_stress, *params_exponnorm)


fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_max_stress, y=cdf_values_norm, mode="lines", name="Normal"))
fig.add_trace(go.Scatter(x=sorted_max_stress, y=cdf_values_lognorm, mode="lines", name="Lognormal"))
fig.add_trace(go.Scatter(x=sorted_max_stress, y=cdf_values_expon, mode="lines", name="Exponential"))
fig.add_trace(go.Scatter(x=sorted_max_stress, y=cdf_values_exponnorm, mode="lines", name="Exponential Normal"))
fig.add_trace(go.Scatter(x=sorted_max_stress, y=cdf, mode="markers", name="Data"))
fig.show()


## Steady State S_xx (normalized)

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Plot cumulative distribution of coupled criterion
weight = np.arange(0, 600, 10)
ss_max_Sxx_norm = df["ss_max_Sxx_norm"][~np.isnan(df["ss_max_Sxx_norm"])]
sorted_ss_max_Sxx_norm = np.sort(ss_max_Sxx_norm)
cdf = np.arange(1, len(sorted_ss_max_Sxx_norm) + 1) / len(sorted_ss_max_Sxx_norm)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_ss_max_Sxx_norm, y=cdf, mode="markers", name="SS MAX SXX NORM", marker=dict(color="red")))
fig.update_layout(xaxis_title="SS MAX SXX NORM", yaxis_title="Cumulative Distribution")
fig.show()


In [ ]:
from scipy import stats

# # Fit a normal distribution to the data
params_norm = stats.norm.fit(ss_max_Sxx_norm)
cdf_values_norm = stats.norm.cdf(sorted_ss_max_Sxx_norm, *params_norm)

# Fit a log-normal distribution to the data
params_lognorm = stats.lognorm.fit(ss_max_Sxx_norm)
cdf_values_lognorm = stats.lognorm.cdf(sorted_ss_max_Sxx_norm, *params_lognorm)

# # Fit an exponential distribution to the data
params_expon = stats.expon.fit(ss_max_Sxx_norm)
cdf_values_expon = stats.expon.cdf(sorted_ss_max_Sxx_norm, *params_expon)

# Fit an Exponential Normal distribution to the data
params_exponnorm = stats.exponnorm.fit(ss_max_Sxx_norm)
cdf_values_exponnorm = stats.exponnorm.cdf(sorted_ss_max_Sxx_norm, *params_exponnorm)
print(params_exponnorm)


fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_ss_max_Sxx_norm, y=cdf_values_norm, mode="lines", name="Normal"))
fig.add_trace(go.Scatter(x=sorted_ss_max_Sxx_norm, y=cdf_values_lognorm, mode="lines", name="Lognormal"))
fig.add_trace(go.Scatter(x=sorted_ss_max_Sxx_norm, y=cdf_values_expon, mode="lines", name="Exponential"))
fig.add_trace(go.Scatter(x=sorted_ss_max_Sxx_norm, y=cdf_values_exponnorm, mode="lines", name="Exponential Normal"))
fig.add_trace(go.Scatter(x=sorted_ss_max_Sxx_norm, y=cdf, mode="markers", name="Data"))
fig.show()


## Slab Tensile Criterion

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Plot cumulative distribution of coupled criterion
weight = np.arange(0, 600, 10)
slab_tensile_criterion = df["slab_tensile_criterion"][~np.isnan(df["slab_tensile_criterion"])]
sorted_slab_tensile_criterion = np.sort(slab_tensile_criterion)
cdf = np.arange(1, len(sorted_slab_tensile_criterion) + 1) / len(sorted_slab_tensile_criterion)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_slab_tensile_criterion, y=cdf, mode="markers", name="SLAB TENSILE CRITERION", marker=dict(color="red")))
fig.update_layout(xaxis_title="SLAB TENSILE CRITERION", yaxis_title="Cumulative Distribution")
fig.show()


In [ ]:
from scipy import stats

# # Fit a normal distribution to the data
params_norm = stats.norm.fit(slab_tensile_criterion)
cdf_values_norm = stats.norm.cdf(sorted_slab_tensile_criterion, *params_norm)

# Fit a log-normal distribution to the data
params_lognorm = stats.lognorm.fit(slab_tensile_criterion)
cdf_values_lognorm = stats.lognorm.cdf(sorted_slab_tensile_criterion, *params_lognorm)
print(params_lognorm)

# # Fit an exponential distribution to the data
params_expon = stats.expon.fit(slab_tensile_criterion)
cdf_values_expon = stats.expon.cdf(sorted_slab_tensile_criterion, *params_expon)

# Fit an Exponential Normal distribution to the data
params_exponnorm = stats.exponnorm.fit(slab_tensile_criterion)
cdf_values_exponnorm = stats.exponnorm.cdf(sorted_slab_tensile_criterion, *params_exponnorm)


fig = go.Figure()
fig.add_trace(go.Scatter(x=sorted_slab_tensile_criterion, y=cdf_values_norm, mode="lines", name="Normal"))
fig.add_trace(go.Scatter(x=sorted_slab_tensile_criterion, y=cdf_values_lognorm, mode="lines", name="Lognormal"))
fig.add_trace(go.Scatter(x=sorted_slab_tensile_criterion, y=cdf_values_expon, mode="lines", name="Exponential"))
fig.add_trace(go.Scatter(x=sorted_slab_tensile_criterion, y=cdf_values_exponnorm, mode="lines", name="Exponential Normal"))
fig.add_trace(go.Scatter(x=sorted_slab_tensile_criterion, y=cdf, mode="markers", name="Data"))
fig.show()
